# Predicción de Precios de Viviendas con Regresión Lineal

**Machine Learning I · UFHEC**

Autor: Feibert Alirio Guzmán Pérez

Este notebook reproduce el flujo completo del proyecto: carga, limpieza, análisis exploratorio, regresión lineal simple, regresión lineal múltiple y evaluación. Las celdas están ordenadas para ejecutarse de arriba hacia abajo.

## 1. Contexto

El dataset **California Housing** contiene información agregada por bloque censal de California (1990). La variable objetivo es `median_house_value`, el valor mediano de la vivienda. Se usan ocho variables numéricas: ubicación, edad, habitaciones, dormitorios, población, hogares e ingreso mediano.

> El objetivo es académico: interpretar relaciones y comunicar resultados, no producir un avalúo.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", palette="viridis")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 2. Carga de datos

El notebook busca `housing.csv` en la carpeta actual y, si no lo encuentra, en la carpeta padre. Así funciona tanto desde la raíz del proyecto como desde `notebooks/`.

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "housing.csv").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "housing.csv"
FIGURES = ROOT / "assets" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Archivo: {DATA_PATH}")
print(f"Filas: {df.shape[0]:,} · Columnas: {df.shape[1]}")
df.head()

## 3. Revisión inicial y calidad de datos

In [ ]:
df.info()
df.describe().T

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

`total_bedrooms` presenta 207 valores nulos. Se imputan con la mediana para conservar todos los registros. `ocean_proximity` es categórica y se excluye del modelo lineal.

In [ ]:
df_clean = df.copy()
df_clean["total_bedrooms"] = df_clean["total_bedrooms"].fillna(
    df_clean["total_bedrooms"].median()
)
df_clean = df_clean.drop(columns=["ocean_proximity"])
print(f"Nulos restantes: {int(df_clean.isna().sum().sum())}")
print(f"Filas conservadas: {len(df_clean):,}")

## 4. Análisis exploratorio

Se revisan la distribución del precio, la correlación entre variables y el patrón geográfico.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df_clean["median_house_value"], bins=50, color="#14B8A6", ax=ax)
ax.set_title("Distribución del valor mediano de la vivienda")
ax.set_xlabel("USD")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

In [ ]:
numeric = df_clean.select_dtypes(include="number")
corr = numeric.corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, linewidths=0.5)
plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()

corr["median_house_value"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(8, 7))
points = plt.scatter(
    df_clean["longitude"],
    df_clean["latitude"],
    c=df_clean["median_house_value"],
    s=df_clean["population"] / 100,
    cmap="viridis",
    alpha=0.45,
)
plt.colorbar(points, label="Valor mediano (USD)")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.title("Distribución geográfica del precio")
plt.tight_layout()
plt.show()

## 5. Regresión lineal simple

Se estima el precio usando únicamente `median_income`. La ecuación resultante es:

`median_house_value = 45,085.58 + 41,793.85 × median_income`.

In [ ]:
X_simple = df_clean[["median_income"]]
y = df_clean["median_house_value"]

simple = LinearRegression().fit(X_simple, y)
print(f"Intercepto: {simple.intercept_:,.2f}")
print(f"Coeficiente: {simple.coef_[0]:,.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(
    data=df_clean,
    x="median_income",
    y="median_house_value",
    scatter_kws={"alpha": 0.2, "s": 10},
    line_kws={"color": "#FB7185"},
    ax=ax,
)
ax.set_title("Ingreso mediano frente a valor mediano")
ax.set_xlabel("Ingreso mediano (decenas de miles de USD)")
ax.set_ylabel("Valor mediano (USD)")
plt.tight_layout()
plt.show()

El coeficiente indica que, por cada aumento de una unidad en `median_income` (10,000 dólares), el modelo estima un incremento promedio de 41,793.85 dólares en el valor de la vivienda. Es una asociación promedio, no un efecto causal.

## 6. Regresión lineal múltiple

Se incorporan las ocho variables numéricas y se evalúa sobre un 20% de datos no vistos.

In [ ]:
features = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
]
X = df_clean[features]
y = df_clean["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

multiple = LinearRegression().fit(X_train, y_train)
prediction = multiple.predict(X_test)
print(f"Entrenamiento: {len(X_train):,} filas")
print(f"Prueba: {len(X_test):,} filas")

In [ ]:
mse = mean_squared_error(y_test, prediction)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, prediction)
r2 = r2_score(y_test, prediction)

metrics = pd.Series(
    {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}
)
metrics

In [ ]:
coefficients = pd.Series(multiple.coef_, index=features).sort_values()
coefficients

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, prediction, alpha=0.25, s=12, color="#14B8A6")
ax.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="#FB7185",
    linewidth=2,
)
ax.set_xlabel("Valor real")
ax.set_ylabel("Predicción")
ax.set_title("Predicción frente a valor real")
plt.tight_layout()
plt.show()

## 7. Interpretación y límites

- El modelo múltiple explica aproximadamente el 61.39% de la variabilidad del precio en prueba.
- El RMSE de 71,133.17 dólares y el MAE de 51,810.48 dólares indican un error relevante para decisiones reales.
- La longitud y la latitud tienen signos negativos porque resumen patrones geográficos correlacionados; no deben interpretarse de forma causal aislada.
- El dataset es agregado, corresponde a 1990 y tiene un techo en 500,001 dólares.
- No se incluyen variables categóricas ni interacciones.

## 8. Conclusiones

El proyecto demuestra que un modelo lineal interpretable puede convertir datos abiertos en una herramienta formativa. La landing y la app Streamlit comunican resultados, métricas y límites. El siguiente paso sería probar codificación de variables categóricas, interacciones y modelos no lineales, manteniendo la evaluación sobre datos no vistos.

## 9. Referencias

- Pace, R. K., & Barry, R. (1997). Sparse spatial autoregressions. *Statistics & Probability Letters*, 33(3), 291-297.
- Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12, 2825-2830.
- scikit-learn Developers. California housing dataset. https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset
- Streamlit. Streamlit documentation. https://docs.streamlit.io